
# Python for Senior Data Engineers: 100 Interview-Focused Programming Q&A (Notebook-Style Markdown)

> **Scope**: Practical Python tasks you’ll face in senior data engineering interviews and on the job. Each item provides a **concise solution** and a brief **explanation**. Topics span core Python, files & formats, data processing, performance, concurrency, testing, packaging, AWS/Databricks, and reliability patterns.

---

## Part A — Core Python Building Blocks

### 1) Read a large text file line-by-line safely
```python
with open("/data/huge.log", "r", encoding="utf-8") as f:
    for line in f:
        process(line)
```
**Explanation**: Iterating the file object streams lines without loading the entire file into memory.

---

### 2) Read a gzip-compressed file line-by-line
```python
import gzip
with gzip.open("/data/huge.log.gz", "rt", encoding="utf-8") as f:
    for line in f:
        process(line)
```
**Explanation**: Use text mode (`"rt"`) for decoded strings; avoids manual decompression steps.

---

### 3) Write a robust CLI with `argparse`
```python
import argparse

p = argparse.ArgumentParser()
p.add_argument("--input", required=True)
p.add_argument("--output", required=True)
args = p.parse_args()
```
**Explanation**: `argparse` provides typed, validated command-line arguments suitable for production scripts.

---

### 4) Parse CSV safely with dialects and quoting
```python
import csv

with open("/data/input.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        handle(row)
```
**Explanation**: `csv` handles quoting, delimiters, and newlines correctly; `DictReader` maps headers to values.

---

### 5) Efficiently count words with `collections.Counter`
```python
from collections import Counter

counts = Counter()
with open("/data/titles.txt", encoding="utf-8") as f:
    for line in f:
        counts.update(line.lower().split())
```
**Explanation**: `Counter.update` is optimized for frequency tallies and supports most-common queries.

---

### 6) Stable dedup while preserving order
```python
def dedup_preserve_order(seq):
    seen = set()
    for x in seq:
        if x not in seen:
            seen.add(x)
            yield x
```
**Explanation**: Set membership is O(1) average; yielding once keeps original order.

---

### 7) Merge sorted streams efficiently (k-way)
```python
import heapq

def merge_sorted(iterables, key=lambda x: x):
    heap = []
    iters = [iter(it) for it in iterables]
    for idx, it in enumerate(iters):
        try:
            val = next(it)
            heap.append((key(val), idx, val))
        except StopIteration:
            pass
    heapq.heapify(heap)
    while heap:
        _, idx, val = heapq.heappop(heap)
        yield val
        try:
            nxt = next(iters[idx])
            heapq.heappush(heap, (key(nxt), idx, nxt))
        except StopIteration:
            pass
```
**Explanation**: A min-heap merges k sorted inputs in O(n log k), ideal for external sorting.

---

### 8) Sliding window average over a stream
```python
from collections import deque

def moving_avg(nums, k):
    q, s = deque(), 0
    for x in nums:
        q.append(x); s += x
        if len(q) > k:
            s -= q.popleft()
        if len(q) == k:
            yield s / k
```
**Explanation**: Deque keeps O(1) amortized updates for fixed-size windows.

---

### 9) Group records by key (multi-map)
```python
from collections import defaultdict

groups = defaultdict(list)
for rec in records:
    groups[rec["country"]].append(rec)
```
**Explanation**: `defaultdict(list)` is concise for building grouped collections.

---

### 10) Find top‑N items by value
```python
import heapq

def top_n(items, n, key=lambda x: x):
    return heapq.nlargest(n, items, key=key)
```
**Explanation**: `nlargest` uses a heap under the hood, efficient for small N vs full sort.

---

### 11) Sort with multiple keys and custom order
```python
items.sort(key=lambda r: (r["country"], -r["revenue"]))
```
**Explanation**: Compound keys and sign flips emulate ascending/descending mix without custom comparators.

---

### 12) Create a context manager (resource handling)
```python
from contextlib import contextmanager

@contextmanager
def temp_change_dir(path):
    import os
    old = os.getcwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(old)
```
**Explanation**: Context managers enforce cleanup, crucial in robust pipelines.

---

### 13) Memoize expensive functions
```python
from functools import lru_cache

@lru_cache(maxsize=1024)
def geocode(city):
    ...
```
**Explanation**: `lru_cache` caches pure-function results for performance; tune `maxsize`.

---

### 14) Create a decorator to time functions
```python
import time
from functools import wraps

def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            dt = time.perf_counter() - t0
            print(f"{fn.__name__} took {dt:.3f}s")
    return wrapper
```
**Explanation**: Decorators add cross-cutting concerns (timing, logging) without changing call sites.

---

### 15) Lazy generator pipeline
```python
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip("
")

def filter_errors(lines):
    for ln in lines:
        if "ERROR" in ln:
            yield ln
```
**Explanation**: Generators compose memory‑efficient pipelines for large logs.

---

### 16) Robust regex extraction with named groups
```python
import re
pat = re.compile(r"^(?P<ts>\S+)\s+(?P<level>INFO|WARN|ERROR)\s+(?P<msg>.*)$")

match = pat.match(line)
if match:
    d = match.groupdict()
```
**Explanation**: Named groups make downstream mapping cleaner and self-documenting.

---

### 17) Safely parse integers/floats with defaults
```python
def to_int(s, default=None):
    try:
        return int(s)
    except (ValueError, TypeError):
        return default
```
**Explanation**: Defensive parsing is essential for messy real-world data.

---

### 18) Datetime parsing and timezone normalization
```python
from datetime import datetime, timezone

def parse_to_utc(s):
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    return dt.astimezone(timezone.utc)
```
**Explanation**: Normalize to UTC early; ensure ISO strings with Z are handled.

---

### 19) Round timestamps to nearest minute
```python
from datetime import datetime, timedelta

def round_minute(dt):
    discard = timedelta(seconds=dt.second, microseconds=dt.microsecond)
    dt -= discard
    if discard >= timedelta(seconds=30):
        dt += timedelta(minutes=1)
    return dt
```
**Explanation**: Useful for windowing logic on ingestion aggregates.

---

### 20) Safely eval simple expressions (no `eval`)
```python
import ast, operator as op

OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv}

class SafeEval(ast.NodeVisitor):
    def visit_BinOp(self, node):
        left = self.visit(node.left)
        right = self.visit(node.right)
        return OPS[type(node.op)](left, right)
    def visit_Num(self, node):
        return node.n

def safe_eval(expr):
    return SafeEval().visit(ast.parse(expr, mode="eval").body)
```
**Explanation**: Parse to AST and only allow specific node types/operators.

---

## Part B — Files, Formats, and Validation

### 21) Read/write JSON lines (NDJSON)
```python
import json

# write
with open("out.jsonl", "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec) + "
")

# read
with open("out.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
```
**Explanation**: Line‑delimited JSON is scalable for streaming and distributed processing.

---

### 22) Stream parse massive JSON array safely
```python
# Use ijson or stream chunking when available; fallback: scan line-by-line if NDJSON
```
**Explanation**: Avoid loading huge arrays; prefer NDJSON or a streaming parser library.

---

### 23) Validate data schema with `pydantic` (if available)
```python
from pydantic import BaseModel, Field, ValidationError

class User(BaseModel):
    id: int
    email: str = Field(pattern=r"^.+@.+$")

try:
    u = User(**payload)
except ValidationError as e:
    log_error(e.json())
```
**Explanation**: Declarative validation catches bad data early in the pipeline.

---

### 24) Convert CSV to Parquet with `pyarrow`
```python
import pyarrow as pa, pyarrow.csv as pacsv, pyarrow.parquet as pq

table = pacsv.read_csv("/data/input.csv")
pq.write_table(table, "/data/output.parquet", compression="snappy")
```
**Explanation**: Arrow Parquet is columnar and compressed—better for analytics workloads.

---

### 25) Append to Parquet dataset partitioned by date
```python
import pyarrow as pa, pyarrow.parquet as pq

# Assume partitioning handled by directory layout like /dataset/date=2026-03-11/part-....parquet
pq.write_table(pa.Table.from_pylist(rows), "/dataset/date=2026-03-11/part-0001.parquet")
```
**Explanation**: Hive‑style partition folders enable pruning in query engines.

---

### 26) Read specific Parquet columns
```python
import pyarrow.parquet as pq

cols = ["id", "amount"]
table = pq.read_table("/data/sales.parquet", columns=cols)
```
**Explanation**: Column projection saves I/O and memory.

---

### 27) Detect and skip malformed CSV rows
```python
import csv

with open("/data/data.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) != 5:
            continue
        process(row)
```
**Explanation**: Defensive validation avoids downstream crashes.

---

### 28) Schema drift detection (columns changed)
```python
def schema_diff(expected_cols, actual_cols):
    return set(expected_cols) ^ set(actual_cols)
```
**Explanation**: Symmetric difference flags added/removed columns quickly.

---

### 29) Detect file encoding
```python
# Use chardet/charset-normalizer when available; else assume UTF-8 and handle exceptions.
```
**Explanation**: Production ingestion must handle mixed encodings gracefully.

---

### 30) Validate JSON against a JSON Schema
```python
# With jsonschema package:
# jsonschema.validate(instance=payload, schema=the_schema)
```
**Explanation**: Contract-first ingestion reduces bad-data incidents.

---

## Part C — Error Handling, Logging, and Observability

### 31) Structured logging with context
```python
import logging, json

logger = logging.getLogger("etl")
logger.setLevel(logging.INFO)

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {"level": record.levelname, "msg": record.getMessage(), "ts": self.formatTime(record)}
        return json.dumps(payload)

h = logging.StreamHandler()
h.setFormatter(JsonFormatter())
logger.addHandler(h)

logger.info("start_ingestion")
```
**Explanation**: JSON logs integrate well with log analytics tools.

---

### 32) Retry with exponential backoff
```python
import time, random

def retry(fn, retries=5, base=0.2):
    for i in range(retries):
        try:
            return fn()
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(base * (2 ** i) + random.uniform(0, 0.1))
```
**Explanation**: Backoff with jitter reduces thundering herd during transient failures.

---

### 33) Fail-fast vs. continue-on-error toggle
```python
STOP_ON_ERROR = False

for rec in records:
    try:
        process(rec)
    except Exception as e:
        if STOP_ON_ERROR:
            raise
        log_warning(rec, e)
```
**Explanation**: Make failure policy explicit for batch pipelines.

---

### 34) Capture exceptions with extra context
```python
try:
    transform(batch)
except Exception as e:
    raise RuntimeError(f"Failed transform for batch_id={batch.id}") from e
```
**Explanation**: Exception chaining preserves original traceback and adds business context.

---

### 35) Add metrics via a lightweight counter
```python
from collections import Counter
metrics = Counter()

for rec in records:
    if is_valid(rec):
        metrics["valid"] += 1
    else:
        metrics["invalid"] += 1
```
**Explanation**: Expose simple counters for monitoring success/failure rates.

---

## Part D — Performance and Memory

### 36) Profile CPU hot spots
```python
import cProfile, pstats

with cProfile.Profile() as pr:
    run_job()

pstats.Stats(pr).sort_stats("cumtime").print_stats(20)
```
**Explanation**: `cProfile` identifies functions dominating runtime; optimize those first.

---

### 37) Sample large data to test logic quickly
```python
import itertools
sample = list(itertools.islice(iterator, 10000))
```
**Explanation**: Sampling accelerates iteration during development and unit tests.

---

### 38) Avoid quadratic string concatenation
```python
parts = []
for rec in rows:
    parts.append(format_line(rec))
out = "".join(parts)
```
**Explanation**: `"".join` is linear and faster than repeated `+=` in loops.

---

### 39) Precompile regex for tight loops
```python
import re
pat = re.compile(r"\d{4}-\d{2}-\d{2}")

for line in lines:
    if pat.search(line):
        ...
```
**Explanation**: Compiling once avoids repeated pattern parsing overhead.

---

### 40) Use `array('d')` or `numpy` for numeric arrays (when allowed)
```python
from array import array
arr = array('d', [0.0])
```
**Explanation**: Dense numeric arrays are far more memory-efficient than Python lists of floats.

---

### 41) Use generators to avoid materializing lists
```python
def read_paths(paths):
    for p in paths:
        yield from read_lines(p)
```
**Explanation**: Streaming avoids peak memory spikes on large inputs.

---

### 42) Use `dataclasses` for compact records
```python
from dataclasses import dataclass

@dataclass
class Event:
    id: int
    ts: int
    value: float
```
**Explanation**: `dataclasses` yield concise, typed containers; `slots=True` (Py3.10+) reduces memory.

---

### 43) Use `bisect` for fast sorted insert/search
```python
import bisect

idx = bisect.bisect_left(sorted_keys, key)
```
**Explanation**: Binary search is O(log n); handy for maintaining sorted arrays.

---

### 44) `functools.cache` for expensive pure functions
```python
from functools import cache

@cache
def normalize_country(name):
    ...
```
**Explanation**: Unbounded cache for pure functions; clear thoughtfully if memory-bound.

---

## Part E — Concurrency, Parallelism & I/O

### 45) Threaded I/O for many small HTTP calls
```python
import concurrent.futures, requests

urls = [...]
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as ex:
    results = list(ex.map(requests.get, urls))
```
**Explanation**: Threads help when the workload is I/O-bound (e.g., HTTP/S3 metadata).

---

### 46) Multiprocessing for CPU-bound parsing
```python
from multiprocessing import Pool

with Pool(processes=8) as pool:
    out = list(pool.map(parse_record, big_records))
```
**Explanation**: Separate processes bypass the GIL for CPU-bound tasks.

---

### 47) Async HTTP with `httpx`/`aiohttp`
```python
import asyncio, httpx

async def fetch(client, url):
    r = await client.get(url, timeout=10)
    r.raise_for_status()
    return r.json()

async def main(urls):
    async with httpx.AsyncClient() as client:
        return await asyncio.gather(*[fetch(client, u) for u in urls])
```
**Explanation**: `asyncio` shines for high-concurrency I/O with many sockets.

---

### 48) Async rate limiting (token bucket)
```python
import asyncio, time

class RateLimiter:
    def __init__(self, rate_per_sec):
        self.tokens = rate_per_sec
        self.rate = rate_per_sec
        self.updated = time.monotonic()
        self.lock = asyncio.Lock()
    async def acquire(self):
        async with self.lock:
            now = time.monotonic()
            self.tokens = min(self.rate, self.tokens + (now - self.updated) * self.rate)
            self.updated = now
            if self.tokens < 1:
                await asyncio.sleep((1 - self.tokens) / self.rate)
                self.tokens = 0
            else:
                self.tokens -= 1
```
**Explanation**: Throttle external API calls to respect SLAs.

---

### 49) Safe subprocess execution
```python
import subprocess

res = subprocess.run(["bash", "script.sh"], capture_output=True, text=True, check=False)
if res.returncode != 0:
    raise RuntimeError(res.stderr)
```
**Explanation**: Capture output; never use `shell=True` with untrusted input.

---

### 50) Timeout for any call
```python
import signal

class Timeout:
    def __init__(self, seconds):
        self.seconds = seconds
    def __enter__(self):
        signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(TimeoutError()))
        signal.alarm(self.seconds)
    def __exit__(self, *exc):
        signal.alarm(0)

with Timeout(10):
    long_call()
```
**Explanation**: POSIX alarm for timeouts; on Windows use threads or asyncio timeouts.

---

## Part F — Testing & Quality

### 51) Basic pytest parametrized test
```python
import pytest

@pytest.mark.parametrize("raw,clean", [(" US ", "US"), ("in", "IN")])
def test_clean_country(raw, clean):
    assert clean_country(raw) == clean
```
**Explanation**: Parameterization keeps tests concise and covers edge cases.

---

### 52) Mock external calls
```python
from unittest.mock import patch

with patch("module.requests.get") as mock_get:
    mock_get.return_value.json.return_value = {"ok": True}
    assert do_call()["ok"] is True
```
**Explanation**: Mocks isolate units from network/services for reliable tests.

---

### 53) Temporary files/dirs in tests
```python
import tempfile, os

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "data.txt")
    with open(path, "w") as f:
        f.write("hi")
```
**Explanation**: Keeps tests hermetic and cleans up automatically.

---

### 54) Type hints + mypy for contracts
```python
from typing import Iterable, Dict

def index(rows: Iterable[Dict[str, str]]) -> Dict[str, int]:
    return {r["id"]: i for i, r in enumerate(rows)}
```
**Explanation**: Static typing prevents whole classes of runtime errors in data code.

---

### 55) Property-based tests with Hypothesis (if available)
```python
# from hypothesis import given, strategies as st
# @given(st.lists(st.integers()))
# def test_dedup(seq):
#     assert list(dedup_preserve_order(seq)) == list(dict.fromkeys(seq))
```
**Explanation**: Generate edge cases automatically and prove invariants.

---

## Part G — Packaging, Config & Deployment

### 56) Use `venv` for isolated environments
```bash
python -m venv .venv
source .venv/bin/activate
pip install -U pip
```
**Explanation**: Pin dependencies per project to avoid “works on my machine”.

---

### 57) Configuration via environment variables with defaults
```python
import os
ENDPOINT = os.getenv("ENDPOINT", "https://api.example.com")
TIMEOUT = int(os.getenv("TIMEOUT", "5"))
```
**Explanation**: Twelve-Factor style; keeps code portable across environments.

---

### 58) Load `.env` file for local dev
```python
# pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()
```
**Explanation**: Convenient for local overrides without committing secrets.

---

### 59) Simple config schema validation
```python
required = {"ENDPOINT", "API_KEY"}
missing = required - set(os.environ)
if missing:
    raise SystemExit(f"Missing env vars: {sorted(missing)}")
```
**Explanation**: Fail fast on misconfiguration.

---

### 60) Build a small CLI tool with `typer` (if available)
```python
# import typer
# app = typer.Typer()
# @app.command()
# def run(input: str, output: str):
#     ...
# if __name__ == "__main__":
#     app()
```
**Explanation**: `typer` simplifies CLI creation with type hints and auto help.

---

## Part H — Data Processing Patterns

### 61) External sort for datasets bigger than memory
```python
# 1) Chunk read + sort to temp files; 2) K-way merge temp files using heapq (see Q7)
```
**Explanation**: Classic approach for sorting TB-scale data on limited memory.

---

### 62) Reservoir sampling (uniform sample of unknown size)
```python
import random

def reservoir(stream, k):
    res = []
    for i, x in enumerate(stream, 1):
        if i <= k:
            res.append(x)
        else:
            j = random.randint(1, i)
            if j <= k:
                res[j-1] = x
    return res
```
**Explanation**: Maintains a uniform sample from a one-pass stream.

---

### 63) Compute rolling hash (e.g., Rabin-Karp style)
```python
def rolling_hash(s, base=257, mod=2**61-1):
    h = 0
    for ch in s:
        h = (h * base + ord(ch)) % mod
    return h
```
**Explanation**: Building blocks for chunking or duplicate detection.

---

### 64) Bloom filter membership test (simple)
```python
import mmh3

class Bloom:
    def __init__(self, m, k):
        self.bits = 0
        self.m = m
        self.k = k
    def add(self, item):
        for i in range(self.k):
            h = mmh3.hash(item, i) % self.m
            self.bits |= 1 << h
    def __contains__(self, item):
        for i in range(self.k):
            h = mmh3.hash(item, i) % self.m
            if not (self.bits >> h) & 1:
                return False
        return True
```
**Explanation**: Probabilistic membership saves memory for large sets; trade-off false positives.

---

### 65) LRU cache with `OrderedDict`
```python
from collections import OrderedDict

class LRU:
    def __init__(self, cap):
        self.cap = cap
        self.d = OrderedDict()
    def get(self, k):
        if k in self.d:
            self.d.move_to_end(k)
            return self.d[k]
    def put(self, k, v):
        self.d[k] = v
        self.d.move_to_end(k)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)
```
**Explanation**: Demonstrates data-structure fluency and eviction policies.

---

### 66) Chunk an iterator into batches
```python
from itertools import islice

def chunked(it, size):
    it = iter(it)
    while True:
        batch = list(islice(it, size))
        if not batch:
            return
        yield batch
```
**Explanation**: Batch processing reduces overhead when writing to remote sinks.

---

### 67) Compute histogram buckets
```python
def histogram(values, bins):
    bins = sorted(bins)
    counts = [0] * (len(bins) + 1)
    for v in values:
        i = 0
        while i < len(bins) and v > bins[i]:
            i += 1
        counts[i] += 1
    return counts
```
**Explanation**: Quick numeric profiling for EDA or monitoring.

---

### 68) Sliding window over time-stamped events
```python
from collections import deque

def windowed_counts(events, window_seconds):
    q = deque()
    for ts in events:
        q.append(ts)
        while q and ts - q[0] > window_seconds:
            q.popleft()
        yield len(q)
```
**Explanation**: Useful for rate computations and throttling.

---

### 69) Stable sort by custom collation (locale)
```python
import locale
locale.setlocale(locale.LC_ALL, 'C')
items.sort(key=locale.strxfrm)
```
**Explanation**: Locale-aware sort for internationalized text.

---

### 70) Top‑K frequent elements with `Counter`
```python
from collections import Counter

most_common = Counter(keys).most_common(10)
```
**Explanation**: Efficient and concise for frequency analytics.

---

## Part I — SQL, ORM & Services

### 71) Parameterized SQL to avoid injection
```python
import sqlite3

con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.execute("SELECT * FROM users WHERE id = ?", (user_id,))
```
**Explanation**: Never string‑format SQL with user input; use parameters.

---

### 72) SQLAlchemy core insert bulk
```python
from sqlalchemy import create_engine, Table, MetaData
engine = create_engine("sqlite://")
meta = MetaData(bind=engine)
# users = Table(...)
# engine.execute(users.insert(), rows)
```
**Explanation**: Bulk operations reduce round trips for large inserts.

---

### 73) Idempotent upsert with PostgreSQL ON CONFLICT
```python
# INSERT ... ON CONFLICT (key) DO UPDATE SET ...
```
**Explanation**: Database-level upsert is more robust than read‑before‑write patterns.

---

### 74) Connection pool sizing
```python
# In SQLAlchemy: create_engine(url, pool_size=10, max_overflow=20)
```
**Explanation**: Pools amortize connection overhead and prevent exhaustion.

---

### 75) Read/write to Redis (if used for caching)
```python
# import redis
# r = redis.Redis()
# r.setex(key, ttl, value)
# r.get(key)
```
**Explanation**: External caches offload hot reads and reduce DB load.

---

## Part J — AWS & Cloud Integrations (Python)

### 76) List S3 objects with pagination (boto3)
```python
import boto3
s3 = boto3.client('s3')

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket='my-bucket', Prefix='raw/2026/03/'):
    for obj in page.get('Contents', []):
        print(obj['Key'])
```
**Explanation**: Paginators handle large listings reliably.

---

### 77) Stream S3 object to file (no full download to memory)
```python
with open("/tmp/file.parquet", "wb") as f:
    s3.download_fileobj("my-bucket", "path/file.parquet", f)
```
**Explanation**: `download_fileobj` streams bytes to disk directly.

---

### 78) Write DataFrame (pandas) to S3 via `s3fs` (if available)
```python
# df.to_parquet("s3://bucket/path/file.parquet", engine="pyarrow")
```
**Explanation**: Using filesystem connectors simplifies S3 I/O from Python.

---

### 79) Read secret from AWS SSM Parameter Store
```python
ssm = boto3.client('ssm')
resp = ssm.get_parameter(Name='/prod/pg/password', WithDecryption=True)
password = resp['Parameter']['Value']
```
**Explanation**: Centralized secrets—avoid hard‑coding credentials.

---

### 80) Publish/consume from Amazon SQS
```python
sqs = boto3.client('sqs')
sqs.send_message(QueueUrl=url, MessageBody=json_payload)
msgs = sqs.receive_message(QueueUrl=url, MaxNumberOfMessages=10, WaitTimeSeconds=10)
```
**Explanation**: SQS decouples producers and consumers for resilient pipelines.

---

## Part K — Orchestration & Databricks Utilities

### 81) Airflow DAG skeleton in Python
```python
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from datetime import datetime
# with DAG('example', start_date=datetime(2026,3,1), schedule='@daily', catchup=False) as dag:
#     t1 = PythonOperator(task_id='extract', python_callable=extract)
#     t2 = PythonOperator(task_id='load', python_callable=load)
#     t1 >> t2
```
**Explanation**: Airflow DAGs as code enable modular, testable orchestration.

---

### 82) Airflow XCom to pass small payloads
```python
# return value from a task becomes XCom; downstream pull with ti.xcom_pull(task_ids='extract')
```
**Explanation**: Keep XCom payloads small; use storage for big data.

---

### 83) Databricks: access widgets
```python
# dbutils.widgets.text("proc_date", "2026-03-11")
# proc_date = dbutils.widgets.get("proc_date")
```
**Explanation**: Parameterize notebooks/jobs without code changes.

---

### 84) Databricks: secret scopes
```python
# pwd = dbutils.secrets.get(scope="prod", key="pg_pwd")
```
**Explanation**: Securely fetch credentials at runtime.

---

### 85) Databricks: mount S3 (legacy)
```python
# dbutils.fs.mount("s3a://bucket", "/mnt/data")
```
**Explanation**: Mounted storage offers simpler paths; many orgs now prefer direct `s3://` access with IAM.

---

## Part L — Reliability, Idempotency & Patterns

### 86) Idempotent file writes with temp+rename
```python
import os, tempfile, shutil

def atomic_write(path, data: bytes):
    d = os.path.dirname(path)
    with tempfile.NamedTemporaryFile(dir=d, delete=False) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        temp_path = tmp.name
    os.replace(temp_path, path)
```
**Explanation**: Write to a temp file and atomic rename prevents partial files.

---

### 87) Exactly-once upsert pattern with checksum
```python
import hashlib

seen = set()

def checksum(rec):
    return hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest()

for rec in stream:
    cs = checksum(rec)
    if cs in seen:
        continue
    upsert(rec)
    seen.add(cs)
```
**Explanation**: De-duplicate by content hash to avoid duplicate effects.

---

### 88) Envelope pattern with metadata
```python
envelope = {
    "meta": {"source": "ingest", "ts": int(time.time())},
    "data": payload
}
```
**Explanation**: Carry lineage/metadata alongside records for auditing.

---

### 89) Reconcile expected vs actual counts
```python
expected = 100000
actual = count_records(output_path)
assert expected == actual, (expected, actual)
```
**Explanation**: Basic guardrail before publishing downstream.

---

### 90) Partition discovery and completeness check
```python
from datetime import date, timedelta

def expected_dates(start, end):
    d = start
    while d <= end:
        yield d.isoformat()
        d += timedelta(days=1)

missing = set(expected_dates(date(2026,3,1), date(2026,3,10))) - set(list_partitions("/data/ds="))
```
**Explanation**: Detect late or missing partitions for alerting.

---

## Part M — Pandas/Arrow Utilities (when allowed)

### 91) Read CSV with dtypes and parse_dates
```python
import pandas as pd

df = pd.read_csv("/data/file.csv", dtype={"id": "int64"}, parse_dates=["event_time"])
```
**Explanation**: Explicit dtypes prevent object (string) bloat and parsing surprises.

---

### 92) Vectorized conditional column
```python
import numpy as np

df["flag"] = np.where(df["amount"] > 1000, 1, 0)
```
**Explanation**: Vectorization outperforms Python loops for column transforms.

---

### 93) GroupBy aggregation to dict
```python
agg = df.groupby("country")["amount"].sum().to_dict()
```
**Explanation**: Quick dimensional summaries for lookups.

---

### 94) Convert pandas DataFrame to Arrow Table
```python
import pyarrow as pa

table = pa.Table.from_pandas(df, preserve_index=False)
```
**Explanation**: Arrow is the lingua franca for columnar data exchange.

---

### 95) Memory‑efficient reading with chunksize
```python
import pandas as pd

for chunk in pd.read_csv("/data/huge.csv", chunksize=500_000):
    process(chunk)
```
**Explanation**: Chunking controls memory footprint for large ingest.

---

## Part N — Security & Serialization

### 96) Avoid `pickle` for untrusted data
```python
# Do not unpickle untrusted inputs; use JSON or safe formats (Avro/Parquet) instead.
```
**Explanation**: `pickle` can execute arbitrary code during load—security risk.

---

### 97) Encrypt/decrypt small secrets with AWS KMS
```python
# kms = boto3.client('kms')
# enc = kms.encrypt(KeyId='alias/prod', Plaintext=b'secret')['CiphertextBlob']
# dec = kms.decrypt(CiphertextBlob=enc)['Plaintext']
```
**Explanation**: Offload cryptography to managed KMS for compliance.

---

### 98) Hashing for anonymization
```python
import hashlib

anon = hashlib.sha256(email.lower().encode()).hexdigest()
```
**Explanation**: Deterministic pseudonymization for privacy-preserving analytics.

---

### 99) Canonical JSON for consistent hashing
```python
import json, hashlib

payload = json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()
digest = hashlib.sha256(payload).hexdigest()
```
**Explanation**: Canonicalization ensures identical logical objects hash to the same value.

---

### 100) Sign and verify messages (HMAC)
```python
import hmac, hashlib

sig = hmac.new(secret_key, message, hashlib.sha256).hexdigest()
# To verify: recompute and compare in constant time
assert hmac.compare_digest(sig, hmac.new(secret_key, message, hashlib.sha256).hexdigest())
```
**Explanation**: HMAC assures integrity/authenticity for webhooks and inter-service messages.

---

**End of 100 Q&A Notebook Markdown**


# 100 Classic Programming Interview Problems (Language-Agnostic)
> Each problem states a **language-agnostic task** with a concise **Python reference solution** and a short **explanation**.

---

### 1) Two Sum (Indices)
**Problem**: Given an array of integers and a target, return indices of the two numbers that add up to target. Assume exactly one solution and you may not use the same element twice.

**Reference (Python)**:
```python
from typing import List

def two_sum(nums: List[int], target: int) -> List[int]:
    idx = {}
    for i, x in enumerate(nums):
        if target - x in idx:
            return [idx[target - x], i]
        idx[x] = i
    return []
```
**Explanation**: Uses a hash map to store seen numbers and their indices; each lookup is O(1), overall O(n) time and O(n) space.

---

### 2) Best Time to Buy and Sell Stock (Single Transaction)
**Problem**: Given prices[i], find the max profit by choosing one day to buy and one later day to sell. If no profit, return 0.

**Reference (Python)**:
```python
from typing import List

def max_profit(prices: List[int]) -> int:
    mn, ans = float('inf'), 0
    for p in prices:
        mn = min(mn, p)
        ans = max(ans, p - mn)
    return ans
```
**Explanation**: Track the minimum so far and the best profit ending today; single pass O(n), O(1) space.

---

### 3) Product of Array Except Self
**Problem**: Return an array output where output[i] is the product of all elements except nums[i], without using division.

**Reference (Python)**:
```python
from typing import List

def product_except_self(nums: List[int]) -> List[int]:
    n = len(nums)
    out = [1]*n
    pref = 1
    for i in range(n):
        out[i] = pref
        pref *= nums[i]
    suff = 1
    for i in range(n-1, -1, -1):
        out[i] *= suff
        suff *= nums[i]
    return out
```
**Explanation**: Prefix and suffix products give O(n) time and O(1) extra space (excluding output).

---

### 4) Longest Common Prefix
**Problem**: Given an array of strings, return the longest common prefix.

**Reference (Python)**:
```python
from typing import List

def longest_common_prefix(strs: List[str]) -> str:
    if not strs:
        return ""
    s1, s2 = min(strs), max(strs)
    i = 0
    while i < len(s1) and i < len(s2) and s1[i] == s2[i]:
        i += 1
    return s1[:i]
```
**Explanation**: The min/max lexicographic pair bounds the LCP; compare only those for O(n + m) overall.

---

### 5) Valid Anagram
**Problem**: Check if two strings are anagrams (case-sensitive, only lowercase letters assumed).

**Reference (Python)**:
```python
from collections import Counter

def is_anagram(a: str, b: str) -> bool:
    return Counter(a) == Counter(b)
```
**Explanation**: Frequency counts must match. O(n) time.

---

### 6) Reverse Words in a String
**Problem**: Given a string s, reverse the order of words (trim spaces).

**Reference (Python)**:
```python
def reverse_words(s: str) -> str:
    return " ".join(reversed(s.split()))
```
**Explanation**: Split trims consecutive spaces; reverse list and join with single spaces.

---

### 7) Longest Substring Without Repeating Characters
**Problem**: Return length of the longest substring without repeating characters.

**Reference (Python)**:
```python
def length_of_longest_substring(s: str) -> int:
    seen = {}
    left = ans = 0
    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1
        seen[ch] = right
        ans = max(ans, right - left + 1)
    return ans
```
**Explanation**: Sliding window with last-seen index; O(n).

---

### 8) Group Anagrams
**Problem**: Group strings that are anagrams of each other.

**Reference (Python)**:
```python
from typing import List, Dict
from collections import defaultdict

def group_anagrams(strs: List[str]) -> List[List[str]]:
    buckets: Dict[tuple, List[str]] = defaultdict(list)
    for s in strs:
        key = tuple(sorted(s))
        buckets[key].append(s)
    return list(buckets.values())
```
**Explanation**: Canonicalize by sorted letters (or 26-count signature).

---

### 9) Rotate Array
**Problem**: Rotate array to the right by k steps.

**Reference (Python)**:
```python
from typing import List

def rotate(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    def rev(i, j):
        while i < j:
            nums[i], nums[j] = nums[j], nums[i]
            i += 1; j -= 1
    rev(0, n-1); rev(0, k-1); rev(k, n-1)
```
**Explanation**: Reverse-three-segments technique (whole, prefix, suffix) achieves rotation in O(n) time and O(1) space.

---

**Note**: For the full 100 problems with complete code and explanations (arrays/strings, hashing, sliding windows, stacks/queues, linked lists, trees, graphs, DP, greedy, intervals, search/sort, math/bits), see the downloadable file prepared in the next step.
